In [0]:
WITH
silver_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT record_hash) AS distinct_record_hash_count,
    LEAST(
      MIN(TO_DATE(pickup_datetime)),
      MIN(TO_DATE(dropoff_datetime))
    ) AS minimum_source_date,
    GREATEST(
      MAX(TO_DATE(pickup_datetime)),
      MAX(TO_DATE(dropoff_datetime))
    ) AS maximum_source_date,
    COUNT_IF(_has_dq_warnings) AS records_with_warning,
    COUNT_IF(_requires_data_review) AS records_requiring_review,
    COUNT_IF(_eligibility_status = 'PARTIALLY_ELIGIBLE')
      AS partially_eligible_records,
    COUNT_IF(_eligibility_status = 'REVIEW_REQUIRED')
      AS review_required_records,
    COUNT_IF(_is_financially_unreconciled)
      AS financially_unreconciled_records,
    COUNT_IF(_is_negative_total_amount) AS negative_total_amount_records,
    COUNT_IF(_is_zero_distance) AS zero_distance_records,
    COUNT_IF(_is_cross_year_trip) AS cross_year_records,
    COUNT_IF(NOT _is_efficiency_metric_eligible)
      AS efficiency_ineligible_records
  FROM nyc_taxi.silver.silver_yellow_trip_2025
),

silver_zone_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT location_id) AS distinct_location_count
  FROM nyc_taxi.silver.silver_taxi_zone_lookup
),

calendar_expectations AS (
  SELECT
    minimum_source_date,
    maximum_source_date,
    YEAR(minimum_source_date) AS minimum_source_year,
    YEAR(maximum_source_date) AS maximum_source_year,
    MAKE_DATE(YEAR(minimum_source_date) - 1, 1, 1)
      AS expected_calendar_start_date,
    MAKE_DATE(YEAR(maximum_source_date) + 1, 12, 31)
      AS expected_calendar_end_date,
    DATEDIFF(
      MAKE_DATE(YEAR(maximum_source_date) + 1, 12, 31),
      MAKE_DATE(YEAR(minimum_source_date) - 1, 1, 1)
    ) + 2 AS expected_calendar_row_count
  FROM silver_stats
),

fact_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT trip_key) AS distinct_trip_key_count,
    COUNT(DISTINCT source_record_hash) AS distinct_source_hash_count,
    COUNT_IF(
      trip_key IS NULL
      OR TRIM(trip_key) = ''
      OR source_record_hash IS NULL
      OR TRIM(source_record_hash) = ''
    ) AS missing_business_key_count,
    COUNT_IF(NOT (trip_key <=> source_record_hash))
      AS trip_source_hash_mismatch_count,
    COUNT_IF(
      pickup_date_key IS NULL
      OR dropoff_date_key IS NULL
      OR pickup_time_key IS NULL
      OR dropoff_time_key IS NULL
      OR pickup_zone_key IS NULL
      OR dropoff_zone_key IS NULL
      OR payment_type_key IS NULL
      OR rate_code_key IS NULL
      OR vendor_key IS NULL
    ) AS null_dimension_key_count,
    COUNT_IF(pickup_date_key = 0 OR dropoff_date_key = 0)
      AS unknown_date_dimension_count,
    COUNT_IF(
      pickup_date_key = 0
      OR dropoff_date_key = 0
      OR pickup_time_key = -1
      OR dropoff_time_key = -1
      OR pickup_zone_key = 0
      OR dropoff_zone_key = 0
    ) AS unknown_critical_dimension_count,
    COUNT_IF(
      payment_type_key = -1
      OR rate_code_key = -1
      OR vendor_key = -1
    ) AS unknown_business_dimension_count,
    COUNT_IF(trip_count IS NULL OR trip_count <> 1)
      AS invalid_trip_count_count,
    COUNT_IF(
      route_key IS NULL
      OR route_key <> CONCAT(
        CAST(pickup_zone_key AS STRING),
        ':',
        CAST(dropoff_zone_key AS STRING)
      )
    ) AS invalid_route_key_count,
    COUNT_IF(
      pickup_datetime IS NULL
      OR dropoff_datetime IS NULL
      OR trip_duration_minutes IS NULL
      OR trip_duration_minutes <= 0
    ) AS invalid_temporal_measure_count,
    COUNT_IF(
      dq_warning_reasons IS NULL
      OR dq_warning_count IS NULL
      OR has_dq_warnings IS NULL
      OR dq_warning_count <> SIZE(dq_warning_reasons)
      OR NOT (has_dq_warnings <=> (SIZE(dq_warning_reasons) > 0))
    ) AS warning_contract_mismatch_count,
    COUNT_IF(
      eligibility_reasons IS NULL
      OR eligibility_restriction_count IS NULL
      OR eligibility_status IS NULL
      OR eligibility_restriction_count <> SIZE(eligibility_reasons)
      OR eligibility_status NOT IN (
        'FULLY_ELIGIBLE',
        'PARTIALLY_ELIGIBLE',
        'REVIEW_REQUIRED'
      )
      OR NOT (
        (eligibility_status = 'REVIEW_REQUIRED')
        <=> requires_data_review
      )
    ) AS eligibility_contract_mismatch_count,
    COUNT_IF(
      quality_rule_version IS NULL
      OR quality_rule_version <> '2.0'
      OR silver_refresh_timestamp IS NULL
      OR eligibility_refresh_timestamp IS NULL
      OR gold_refresh_timestamp IS NULL
    ) AS invalid_quality_metadata_count,
    COUNT_IF(
      NOT COALESCE(is_trip_volume_metric_eligible, FALSE)
      OR NOT COALESCE(is_recorded_amount_metric_eligible, FALSE)
      OR NOT COALESCE(is_route_metric_eligible, FALSE)
    ) AS mandatory_metric_ineligible_count,
    COUNT_IF(
      NOT (
        is_efficiency_metric_eligible
        <=> (
          is_distance_metric_eligible
          AND is_duration_metric_eligible
        )
      )
      OR NOT (
        is_efficiency_kpi_eligible
        <=> is_efficiency_metric_eligible
      )
      OR NOT (
        is_ml_standard_trip_eligible
        <=> (
          is_standard_operational_trip_eligible
          AND is_passenger_metric_eligible
          AND is_ml_financial_feature_eligible
          AND is_ml_categorical_feature_eligible
        )
      )
    ) AS derived_eligibility_mismatch_count,
    COUNT_IF(
      NOT (
        (payment_type_key = -1)
        <=> (is_payment_type_metric_eligible = FALSE)
      )
      OR NOT (
        (rate_code_key = -1)
        <=> (is_rate_code_metric_eligible = FALSE)
      )
      OR NOT (
        (vendor_key = -1)
        <=> (is_vendor_metric_eligible = FALSE)
      )
    ) AS business_dimension_eligibility_mismatch_count,
    COUNT_IF(
      NOT (
        is_financially_unreconciled
        <=> (ABS(financial_reconciliation_difference) > 0.01)
      )
      OR NOT (
        financial_reconciliation_difference
        <=> CAST(
          total_amount - financial_component_amount
          AS DECIMAL(20, 2)
        )
      )
    ) AS financial_contract_mismatch_count,
    COUNT_IF(
      NOT (
        reported_card_tip_amount
        <=> CASE
          WHEN payment_type_key = 1
               AND is_reported_tip_metric_eligible
            THEN tip_amount
          ELSE CAST(0 AS DECIMAL(18, 2))
        END
      )
      OR NOT (
        has_reported_electronic_tip
        <=> (
          payment_type_key = 1
          AND is_reported_tip_metric_eligible
          AND tip_amount > 0
        )
      )
    ) AS reported_tip_contract_mismatch_count,
    COUNT_IF(has_dq_warnings) AS records_with_warning,
    COUNT_IF(requires_data_review) AS records_requiring_review,
    COUNT_IF(eligibility_status = 'PARTIALLY_ELIGIBLE')
      AS partially_eligible_records,
    COUNT_IF(eligibility_status = 'REVIEW_REQUIRED')
      AS review_required_records,
    COUNT_IF(is_financially_unreconciled)
      AS financially_unreconciled_records,
    COUNT_IF(is_negative_total_amount) AS negative_total_amount_records,
    COUNT_IF(is_zero_distance) AS zero_distance_records,
    COUNT_IF(is_cross_year_trip) AS cross_year_records,
    COUNT_IF(NOT is_efficiency_metric_eligible)
      AS efficiency_ineligible_records,
    COUNT_IF(NOT is_financial_breakdown_metric_eligible)
      AS financial_breakdown_ineligible_records,
    COUNT_IF(NOT is_ml_standard_trip_eligible)
      AS ml_standard_trip_ineligible_records,
    MIN(TO_DATE(pickup_datetime)) AS minimum_pickup_date,
    MAX(TO_DATE(pickup_datetime)) AS maximum_pickup_date,
    MIN(TO_DATE(dropoff_datetime)) AS minimum_dropoff_date,
    MAX(TO_DATE(dropoff_datetime)) AS maximum_dropoff_date
  FROM nyc_taxi.gold.fact_yellow_taxi_trip
),

contract_parity AS (
  SELECT
    COUNT_IF(s.record_hash IS NULL) AS fact_without_silver_record_count,
    COUNT_IF(
      s.record_hash IS NOT NULL
      AND (
        NOT (f.dq_warning_reasons <=> s._dq_warning_reasons)
        OR NOT (f.dq_warning_count <=> s._dq_warning_count)
        OR NOT (f.has_dq_warnings <=> s._has_dq_warnings)
      )
    ) AS warning_propagation_mismatch_count,
    COUNT_IF(
      s.record_hash IS NOT NULL
      AND (
        NOT (f.eligibility_reasons <=> s._eligibility_reasons)
        OR NOT (
          f.eligibility_restriction_count
          <=> s._eligibility_restriction_count
        )
        OR NOT (f.eligibility_status <=> s._eligibility_status)
        OR NOT (
          f.requires_data_review
          <=> s._requires_data_review
        )
      )
    ) AS eligibility_propagation_mismatch_count,
    COUNT_IF(
      s.record_hash IS NOT NULL
      AND (
        NOT (
          f.is_trip_volume_metric_eligible
          <=> s._is_trip_volume_metric_eligible
        )
        OR NOT (
          f.is_recorded_amount_metric_eligible
          <=> s._is_recorded_amount_metric_eligible
        )
        OR NOT (
          f.is_efficiency_metric_eligible
          <=> s._is_efficiency_metric_eligible
        )
        OR NOT (
          f.is_financial_breakdown_metric_eligible
          <=> s._is_financial_breakdown_metric_eligible
        )
        OR NOT (
          f.is_ml_standard_trip_eligible
          <=> s._is_ml_standard_trip_eligible
        )
      )
    ) AS eligibility_flag_propagation_mismatch_count,
    COUNT_IF(
      s.record_hash IS NOT NULL
      AND (
        NOT (
          f.financial_reconciliation_difference
          <=> s._financial_reconciliation_difference
        )
        OR NOT (
          f.is_financially_unreconciled
          <=> s._is_financially_unreconciled
        )
      )
    ) AS financial_propagation_mismatch_count
  FROM nyc_taxi.gold.fact_yellow_taxi_trip f
  LEFT JOIN nyc_taxi.silver.silver_yellow_trip_2025 s
    ON f.source_record_hash = s.record_hash
),

date_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT date_key) AS distinct_key_count,
    COUNT_IF(date_key = 0) AS unknown_member_count,
    MIN(CASE WHEN date_key <> 0 THEN full_date END) AS minimum_calendar_date,
    MAX(CASE WHEN date_key <> 0 THEN full_date END) AS maximum_calendar_date,
    COUNT_IF(
      (date_key = 0 AND (
        full_date IS NOT NULL
        OR calendar_year_role <> 'UNKNOWN'
        OR is_source_year
      ))
      OR (date_key <> 0 AND (
        full_date IS NULL
        OR date_key <> CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT)
        OR calendar_year_role NOT IN (
          'PREDECESSOR_YEAR',
          'SOURCE_YEAR_RANGE',
          'SUCCESSOR_YEAR'
        )
      ))
    ) AS invalid_member_count,
    COUNT_IF(
      date_key <> 0
      AND NOT (
        calendar_year_role
        <=> CASE
          WHEN YEAR(full_date) < bounds.minimum_source_year
            THEN 'PREDECESSOR_YEAR'
          WHEN YEAR(full_date) > bounds.maximum_source_year
            THEN 'SUCCESSOR_YEAR'
          ELSE 'SOURCE_YEAR_RANGE'
        END
      )
    ) AS invalid_year_role_count,
    COUNT_IF(
      date_key <> 0
      AND NOT (
        is_source_year
        <=> (
          YEAR(full_date) BETWEEN
            bounds.minimum_source_year
            AND bounds.maximum_source_year
        )
      )
    ) AS invalid_source_year_flag_count,
    COUNT_IF(
      date_key <> 0
      AND (
        NOT (is_weekend <=> (DAYOFWEEK(full_date) IN (1, 7)))
        OR NOT (is_business_day <=> (NOT is_weekend AND holiday_name IS NULL))
        OR (
          holiday_name IS NOT NULL
          AND NOT (
            is_federal_holiday
            AND holiday_scope = 'US Federal'
          )
        )
      )
    ) AS invalid_calendar_attribute_count
  FROM nyc_taxi.gold.dim_date
  CROSS JOIN calendar_expectations bounds
),

time_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT time_key) AS distinct_key_count,
    COUNT_IF(time_key = -1) AS unknown_member_count,
    COUNT_IF(
      (time_key = -1 AND hour_number IS NOT NULL)
      OR (time_key <> -1 AND (
        hour_number NOT BETWEEN 0 AND 23
        OR time_key <> hour_number
      ))
    ) AS invalid_member_count
  FROM nyc_taxi.gold.dim_time
),

pickup_zone_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT pickup_zone_key) AS distinct_key_count,
    COUNT_IF(pickup_zone_key = 0) AS unknown_member_count
  FROM nyc_taxi.gold.dim_pickup_zone
),

dropoff_zone_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT dropoff_zone_key) AS distinct_key_count,
    COUNT_IF(dropoff_zone_key = 0) AS unknown_member_count
  FROM nyc_taxi.gold.dim_dropoff_zone
),

payment_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT payment_type_key) AS distinct_key_count,
    COUNT_IF(payment_type_key = -1) AS unknown_member_count
  FROM nyc_taxi.gold.dim_payment_type
),

rate_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT rate_code_key) AS distinct_key_count,
    COUNT_IF(rate_code_key = -1) AS unknown_member_count
  FROM nyc_taxi.gold.dim_rate_code
),

vendor_stats AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT vendor_key) AS distinct_key_count,
    COUNT_IF(vendor_key = -1) AS unknown_member_count
  FROM nyc_taxi.gold.dim_vendor
),

foreign_key_stats AS (
  SELECT
    COUNT_IF(pd.date_key IS NULL) AS orphan_pickup_date_count,
    COUNT_IF(dd.date_key IS NULL) AS orphan_dropoff_date_count,
    COUNT_IF(pt.time_key IS NULL) AS orphan_pickup_time_count,
    COUNT_IF(dt.time_key IS NULL) AS orphan_dropoff_time_count,
    COUNT_IF(pz.pickup_zone_key IS NULL) AS orphan_pickup_zone_count,
    COUNT_IF(dz.dropoff_zone_key IS NULL) AS orphan_dropoff_zone_count,
    COUNT_IF(pm.payment_type_key IS NULL) AS orphan_payment_count,
    COUNT_IF(rc.rate_code_key IS NULL) AS orphan_rate_count,
    COUNT_IF(v.vendor_key IS NULL) AS orphan_vendor_count
  FROM nyc_taxi.gold.fact_yellow_taxi_trip f
  LEFT JOIN nyc_taxi.gold.dim_date pd
    ON f.pickup_date_key = pd.date_key
  LEFT JOIN nyc_taxi.gold.dim_date dd
    ON f.dropoff_date_key = dd.date_key
  LEFT JOIN nyc_taxi.gold.dim_time pt
    ON f.pickup_time_key = pt.time_key
  LEFT JOIN nyc_taxi.gold.dim_time dt
    ON f.dropoff_time_key = dt.time_key
  LEFT JOIN nyc_taxi.gold.dim_pickup_zone pz
    ON f.pickup_zone_key = pz.pickup_zone_key
  LEFT JOIN nyc_taxi.gold.dim_dropoff_zone dz
    ON f.dropoff_zone_key = dz.dropoff_zone_key
  LEFT JOIN nyc_taxi.gold.dim_payment_type pm
    ON f.payment_type_key = pm.payment_type_key
  LEFT JOIN nyc_taxi.gold.dim_rate_code rc
    ON f.rate_code_key = rc.rate_code_key
  LEFT JOIN nyc_taxi.gold.dim_vendor v
    ON f.vendor_key = v.vendor_key
),

validation_results AS (
  SELECT
    'VOLUME' AS validation_group,
    'silver_to_gold_record_conservation' AS validation_name,
    'BLOCKER' AS severity,
    CASE WHEN f.row_count = s.row_count THEN 'PASS' ELSE 'FAIL' END AS status,
    ABS(f.row_count - s.row_count) AS exception_record_count,
    s.row_count AS evaluated_record_count,
    CONCAT(
      'silver=', CAST(s.row_count AS STRING),
      '; gold_fact=', CAST(f.row_count AS STRING)
    ) AS details
  FROM fact_stats f CROSS JOIN silver_stats s

  UNION ALL

  SELECT
    'FACT_CONTRACT',
    'business_key_is_present_and_unique',
    'BLOCKER',
    CASE
      WHEN f.row_count > 0
       AND f.missing_business_key_count = 0
       AND f.trip_source_hash_mismatch_count = 0
       AND f.distinct_trip_key_count = f.row_count
       AND f.distinct_source_hash_count = f.row_count
      THEN 'PASS' ELSE 'FAIL'
    END,
    f.missing_business_key_count
      + f.trip_source_hash_mismatch_count
      + (f.row_count - f.distinct_trip_key_count)
      + (f.row_count - f.distinct_source_hash_count),
    f.row_count,
    CONCAT(
      'rows=', CAST(f.row_count AS STRING),
      '; distinct_trip_keys=', CAST(f.distinct_trip_key_count AS STRING),
      '; distinct_source_hashes=', CAST(f.distinct_source_hash_count AS STRING),
      '; missing_keys=', CAST(f.missing_business_key_count AS STRING),
      '; trip_source_mismatches=',
        CAST(f.trip_source_hash_mismatch_count AS STRING)
    )
  FROM fact_stats f

  UNION ALL

  SELECT
    'FACT_CONTRACT',
    'mandatory_measures_and_dimension_keys',
    'BLOCKER',
    CASE
      WHEN f.null_dimension_key_count = 0
       AND f.invalid_trip_count_count = 0
       AND f.invalid_temporal_measure_count = 0
       AND f.invalid_route_key_count = 0
      THEN 'PASS' ELSE 'FAIL'
    END,
    f.null_dimension_key_count
      + f.invalid_trip_count_count
      + f.invalid_temporal_measure_count
      + f.invalid_route_key_count,
    f.row_count,
    CONCAT(
      'null_dimension_keys=', CAST(f.null_dimension_key_count AS STRING),
      '; invalid_trip_count=', CAST(f.invalid_trip_count_count AS STRING),
      '; invalid_temporal=', CAST(f.invalid_temporal_measure_count AS STRING),
      '; invalid_route=', CAST(f.invalid_route_key_count AS STRING)
    )
  FROM fact_stats f

  UNION ALL

  SELECT
    'FACT_CONTRACT',
    'critical_dimensions_are_resolved',
    'BLOCKER',
    CASE WHEN f.unknown_critical_dimension_count = 0
      THEN 'PASS' ELSE 'FAIL' END,
    f.unknown_critical_dimension_count,
    f.row_count,
    CONCAT(
      'unknown_critical_dimensions=',
        CAST(f.unknown_critical_dimension_count AS STRING),
      '; unknown_dates=', CAST(f.unknown_date_dimension_count AS STRING)
    )
  FROM fact_stats f

  UNION ALL

  SELECT
    'REFERENTIAL_INTEGRITY',
    'all_fact_foreign_keys_exist',
    'BLOCKER',
    CASE
      WHEN fk.orphan_pickup_date_count
         + fk.orphan_dropoff_date_count
         + fk.orphan_pickup_time_count
         + fk.orphan_dropoff_time_count
         + fk.orphan_pickup_zone_count
         + fk.orphan_dropoff_zone_count
         + fk.orphan_payment_count
         + fk.orphan_rate_count
         + fk.orphan_vendor_count = 0
      THEN 'PASS' ELSE 'FAIL'
    END,
    fk.orphan_pickup_date_count
      + fk.orphan_dropoff_date_count
      + fk.orphan_pickup_time_count
      + fk.orphan_dropoff_time_count
      + fk.orphan_pickup_zone_count
      + fk.orphan_dropoff_zone_count
      + fk.orphan_payment_count
      + fk.orphan_rate_count
      + fk.orphan_vendor_count,
    f.row_count,
    CONCAT(
      'pickup_date=', CAST(fk.orphan_pickup_date_count AS STRING),
      '; dropoff_date=', CAST(fk.orphan_dropoff_date_count AS STRING),
      '; pickup_time=', CAST(fk.orphan_pickup_time_count AS STRING),
      '; dropoff_time=', CAST(fk.orphan_dropoff_time_count AS STRING),
      '; pickup_zone=', CAST(fk.orphan_pickup_zone_count AS STRING),
      '; dropoff_zone=', CAST(fk.orphan_dropoff_zone_count AS STRING),
      '; payment=', CAST(fk.orphan_payment_count AS STRING),
      '; rate=', CAST(fk.orphan_rate_count AS STRING),
      '; vendor=', CAST(fk.orphan_vendor_count AS STRING)
    )
  FROM foreign_key_stats fk CROSS JOIN fact_stats f

  UNION ALL

  SELECT
    'DIMENSION_CONTRACT',
    'dynamic_date_coverage',
    'BLOCKER',
    CASE
      WHEN d.row_count = c.expected_calendar_row_count
       AND d.distinct_key_count = d.row_count
       AND d.unknown_member_count = 1
       AND d.minimum_calendar_date = c.expected_calendar_start_date
       AND d.maximum_calendar_date = c.expected_calendar_end_date
       AND d.invalid_member_count = 0
       AND d.invalid_year_role_count = 0
       AND d.invalid_source_year_flag_count = 0
      THEN 'PASS' ELSE 'FAIL'
    END,
    ABS(d.row_count - c.expected_calendar_row_count)
      + (d.row_count - d.distinct_key_count)
      + ABS(d.unknown_member_count - 1)
      + d.invalid_member_count
      + d.invalid_year_role_count
      + d.invalid_source_year_flag_count
      + CASE WHEN d.minimum_calendar_date = c.expected_calendar_start_date
          THEN 0 ELSE 1 END
      + CASE WHEN d.maximum_calendar_date = c.expected_calendar_end_date
          THEN 0 ELSE 1 END,
    c.expected_calendar_row_count,
    CONCAT(
      'source_range=', CAST(c.minimum_source_date AS STRING),
      '..', CAST(c.maximum_source_date AS STRING),
      '; expected_calendar=', CAST(c.expected_calendar_start_date AS STRING),
      '..', CAST(c.expected_calendar_end_date AS STRING),
      '; actual_calendar=', CAST(d.minimum_calendar_date AS STRING),
      '..', CAST(d.maximum_calendar_date AS STRING),
      '; actual_rows=', CAST(d.row_count AS STRING),
      '; expected_rows=', CAST(c.expected_calendar_row_count AS STRING)
    )
  FROM date_stats d CROSS JOIN calendar_expectations c

  UNION ALL

  SELECT
    'DIMENSION_CONTRACT',
    'date_business_attributes',
    'BLOCKER',
    CASE WHEN d.invalid_calendar_attribute_count = 0
      THEN 'PASS' ELSE 'FAIL' END,
    d.invalid_calendar_attribute_count,
    d.row_count,
    'Weekend, business-day and populated federal-holiday attributes must agree'
  FROM date_stats d

  UNION ALL

  SELECT
    'DIMENSION_CONTRACT',
    'time_dimension_complete',
    'BLOCKER',
    CASE
      WHEN t.row_count = 25
       AND t.distinct_key_count = 25
       AND t.unknown_member_count = 1
       AND t.invalid_member_count = 0
      THEN 'PASS' ELSE 'FAIL'
    END,
    ABS(t.row_count - 25)
      + (t.row_count - t.distinct_key_count)
      + ABS(t.unknown_member_count - 1)
      + t.invalid_member_count,
    25,
    CONCAT(
      'rows=', CAST(t.row_count AS STRING),
      '; distinct_keys=', CAST(t.distinct_key_count AS STRING),
      '; unknown_members=', CAST(t.unknown_member_count AS STRING),
      '; invalid_members=', CAST(t.invalid_member_count AS STRING)
    )
  FROM time_stats t

  UNION ALL

  SELECT
    'DIMENSION_CONTRACT',
    'zone_dimensions_match_silver',
    'BLOCKER',
    CASE
      WHEN p.row_count = z.row_count + 1
       AND d.row_count = z.row_count + 1
       AND p.distinct_key_count = p.row_count
       AND d.distinct_key_count = d.row_count
       AND p.unknown_member_count = 1
       AND d.unknown_member_count = 1
       AND z.distinct_location_count = z.row_count
      THEN 'PASS' ELSE 'FAIL'
    END,
    ABS(p.row_count - (z.row_count + 1))
      + ABS(d.row_count - (z.row_count + 1))
      + (p.row_count - p.distinct_key_count)
      + (d.row_count - d.distinct_key_count)
      + ABS(p.unknown_member_count - 1)
      + ABS(d.unknown_member_count - 1)
      + (z.row_count - z.distinct_location_count),
    (z.row_count + 1) * 2,
    CONCAT(
      'silver_zones=', CAST(z.row_count AS STRING),
      '; pickup_dim=', CAST(p.row_count AS STRING),
      '; dropoff_dim=', CAST(d.row_count AS STRING),
      '; expected_each=', CAST(z.row_count + 1 AS STRING)
    )
  FROM pickup_zone_stats p
  CROSS JOIN dropoff_zone_stats d
  CROSS JOIN silver_zone_stats z

  UNION ALL

  SELECT
    'DIMENSION_CONTRACT',
    'business_code_dimensions_complete',
    'BLOCKER',
    CASE
      WHEN p.row_count = 8
       AND p.distinct_key_count = 8
       AND p.unknown_member_count = 1
       AND r.row_count = 8
       AND r.distinct_key_count = 8
       AND r.unknown_member_count = 1
       AND v.row_count = 5
       AND v.distinct_key_count = 5
       AND v.unknown_member_count = 1
      THEN 'PASS' ELSE 'FAIL'
    END,
    ABS(p.row_count - 8)
      + (p.row_count - p.distinct_key_count)
      + ABS(p.unknown_member_count - 1)
      + ABS(r.row_count - 8)
      + (r.row_count - r.distinct_key_count)
      + ABS(r.unknown_member_count - 1)
      + ABS(v.row_count - 5)
      + (v.row_count - v.distinct_key_count)
      + ABS(v.unknown_member_count - 1),
    21,
    CONCAT(
      'payment=', CAST(p.row_count AS STRING),
      '; rate=', CAST(r.row_count AS STRING),
      '; vendor=', CAST(v.row_count AS STRING)
    )
  FROM payment_stats p CROSS JOIN rate_stats r CROSS JOIN vendor_stats v

  UNION ALL

  SELECT
    'QUALITY_CONTRACT',
    'warning_and_eligibility_contracts',
    'BLOCKER',
    CASE
      WHEN f.warning_contract_mismatch_count = 0
       AND f.eligibility_contract_mismatch_count = 0
       AND f.invalid_quality_metadata_count = 0
       AND f.mandatory_metric_ineligible_count = 0
       AND f.derived_eligibility_mismatch_count = 0
       AND f.business_dimension_eligibility_mismatch_count = 0
      THEN 'PASS' ELSE 'FAIL'
    END,
    f.warning_contract_mismatch_count
      + f.eligibility_contract_mismatch_count
      + f.invalid_quality_metadata_count
      + f.mandatory_metric_ineligible_count
      + f.derived_eligibility_mismatch_count
      + f.business_dimension_eligibility_mismatch_count,
    f.row_count,
    CONCAT(
      'warning=', CAST(f.warning_contract_mismatch_count AS STRING),
      '; eligibility=', CAST(f.eligibility_contract_mismatch_count AS STRING),
      '; metadata=', CAST(f.invalid_quality_metadata_count AS STRING),
      '; mandatory_metrics=', CAST(f.mandatory_metric_ineligible_count AS STRING),
      '; derived=', CAST(f.derived_eligibility_mismatch_count AS STRING),
      '; business_dimensions=',
        CAST(f.business_dimension_eligibility_mismatch_count AS STRING)
    )
  FROM fact_stats f

  UNION ALL

  SELECT
    'QUALITY_CONTRACT',
    'financial_and_reported_tip_contracts',
    'BLOCKER',
    CASE
      WHEN f.financial_contract_mismatch_count = 0
       AND f.reported_tip_contract_mismatch_count = 0
      THEN 'PASS' ELSE 'FAIL'
    END,
    f.financial_contract_mismatch_count
      + f.reported_tip_contract_mismatch_count,
    f.row_count,
    CONCAT(
      'financial_mismatches=',
        CAST(f.financial_contract_mismatch_count AS STRING),
      '; reported_tip_mismatches=',
        CAST(f.reported_tip_contract_mismatch_count AS STRING)
    )
  FROM fact_stats f

  UNION ALL

  SELECT
    'SILVER_GOLD_PARITY',
    'record_level_quality_contract_propagation',
    'BLOCKER',
    CASE
      WHEN p.fact_without_silver_record_count = 0
       AND p.warning_propagation_mismatch_count = 0
       AND p.eligibility_propagation_mismatch_count = 0
       AND p.eligibility_flag_propagation_mismatch_count = 0
       AND p.financial_propagation_mismatch_count = 0
      THEN 'PASS' ELSE 'FAIL'
    END,
    p.fact_without_silver_record_count
      + p.warning_propagation_mismatch_count
      + p.eligibility_propagation_mismatch_count
      + p.eligibility_flag_propagation_mismatch_count
      + p.financial_propagation_mismatch_count,
    f.row_count,
    CONCAT(
      'missing_silver=', CAST(p.fact_without_silver_record_count AS STRING),
      '; warning=', CAST(p.warning_propagation_mismatch_count AS STRING),
      '; eligibility=',
        CAST(p.eligibility_propagation_mismatch_count AS STRING),
      '; flags=',
        CAST(p.eligibility_flag_propagation_mismatch_count AS STRING),
      '; financial=', CAST(p.financial_propagation_mismatch_count AS STRING)
    )
  FROM contract_parity p CROSS JOIN fact_stats f

  UNION ALL

  SELECT
    'SILVER_GOLD_PARITY',
    'aggregate_quality_distribution_parity',
    'BLOCKER',
    CASE
      WHEN f.records_with_warning = s.records_with_warning
       AND f.records_requiring_review = s.records_requiring_review
       AND f.partially_eligible_records = s.partially_eligible_records
       AND f.review_required_records = s.review_required_records
       AND f.financially_unreconciled_records
          = s.financially_unreconciled_records
       AND f.negative_total_amount_records = s.negative_total_amount_records
       AND f.zero_distance_records = s.zero_distance_records
       AND f.cross_year_records = s.cross_year_records
       AND f.efficiency_ineligible_records = s.efficiency_ineligible_records
      THEN 'PASS' ELSE 'FAIL'
    END,
    ABS(f.records_with_warning - s.records_with_warning)
      + ABS(f.records_requiring_review - s.records_requiring_review)
      + ABS(f.partially_eligible_records - s.partially_eligible_records)
      + ABS(f.review_required_records - s.review_required_records)
      + ABS(
          f.financially_unreconciled_records
          - s.financially_unreconciled_records
        )
      + ABS(f.negative_total_amount_records - s.negative_total_amount_records)
      + ABS(f.zero_distance_records - s.zero_distance_records)
      + ABS(f.cross_year_records - s.cross_year_records)
      + ABS(f.efficiency_ineligible_records - s.efficiency_ineligible_records),
    s.row_count,
    CONCAT(
      'warnings=', CAST(f.records_with_warning AS STRING),
      '; review=', CAST(f.records_requiring_review AS STRING),
      '; partial=', CAST(f.partially_eligible_records AS STRING),
      '; review_status=', CAST(f.review_required_records AS STRING),
      '; financial_gap=', CAST(f.financially_unreconciled_records AS STRING),
      '; negative_total=', CAST(f.negative_total_amount_records AS STRING),
      '; zero_distance=', CAST(f.zero_distance_records AS STRING),
      '; cross_year=', CAST(f.cross_year_records AS STRING)
    )
  FROM fact_stats f CROSS JOIN silver_stats s

  UNION ALL

  SELECT
    'MONITORING',
    'unknown_business_dimensions',
    'WARNING',
    CASE WHEN f.unknown_business_dimension_count = 0
      THEN 'PASS' ELSE 'WARN' END,
    f.unknown_business_dimension_count,
    f.row_count,
    'Unknown payment, rate or vendor members remain measurable but restrict categorical analysis'
  FROM fact_stats f

  UNION ALL

  SELECT
    'MONITORING',
    'financial_reconciliation_gap',
    'WARNING',
    CASE WHEN f.financially_unreconciled_records = 0
      THEN 'PASS' ELSE 'WARN' END,
    f.financially_unreconciled_records,
    f.row_count,
    CONCAT(
      'excluded_from_financial_breakdown=',
        CAST(f.financial_breakdown_ineligible_records AS STRING),
      '; signed total amount remains eligible when allowed by Silver contract'
    )
  FROM fact_stats f

  UNION ALL

  SELECT
    'MONITORING',
    'negative_total_amount',
    'WARNING',
    CASE WHEN f.negative_total_amount_records = 0
      THEN 'PASS' ELSE 'WARN' END,
    f.negative_total_amount_records,
    f.row_count,
    'Potential reversals or adjustments; preserve signed amounts and filter by eligibility for each metric'
  FROM fact_stats f

  UNION ALL

  SELECT
    'MONITORING',
    'zero_distance_and_efficiency_exclusions',
    'WARNING',
    CASE WHEN f.zero_distance_records = 0
      THEN 'PASS' ELSE 'WARN' END,
    f.zero_distance_records,
    f.row_count,
    CONCAT(
      'zero_distance=', CAST(f.zero_distance_records AS STRING),
      '; efficiency_ineligible=',
        CAST(f.efficiency_ineligible_records AS STRING)
    )
  FROM fact_stats f

  UNION ALL

  SELECT
    'MONITORING',
    'records_requiring_review',
    'WARNING',
    CASE WHEN f.records_requiring_review = 0
      THEN 'PASS' ELSE 'WARN' END,
    f.records_requiring_review,
    f.row_count,
    CONCAT(
      'review_required=', CAST(f.review_required_records AS STRING),
      '; partially_eligible=', CAST(f.partially_eligible_records AS STRING),
      '; ml_standard_ineligible=',
        CAST(f.ml_standard_trip_ineligible_records AS STRING)
    )
  FROM fact_stats f

  UNION ALL

  SELECT
    'MONITORING',
    'cross_year_records_are_covered_by_calendar',
    'WARNING',
    CASE
      WHEN f.unknown_date_dimension_count = 0 THEN 'PASS'
      ELSE 'WARN'
    END,
    f.unknown_date_dimension_count,
    f.row_count,
    CONCAT(
      'cross_year_records=', CAST(f.cross_year_records AS STRING),
      '; unresolved_date_keys=', CAST(f.unknown_date_dimension_count AS STRING),
      '; fact_dropoff_max=', CAST(f.maximum_dropoff_date AS STRING)
    )
  FROM fact_stats f
)

SELECT
  validation_group,
  validation_name,
  severity,
  status,
  CAST(exception_record_count AS BIGINT) AS exception_record_count,
  CAST(evaluated_record_count AS BIGINT) AS evaluated_record_count,
  ROUND(
    CASE
      WHEN evaluated_record_count = 0 THEN 0.0
      ELSE 100.0 * exception_record_count / evaluated_record_count
    END,
    6
  ) AS exception_rate_pct,
  details
FROM validation_results
ORDER BY
  CASE severity
    WHEN 'BLOCKER' THEN 1
    WHEN 'WARNING' THEN 2
    ELSE 3
  END,
  CASE status
    WHEN 'FAIL' THEN 1
    WHEN 'WARN' THEN 2
    WHEN 'PASS' THEN 3
    ELSE 4
  END,
  validation_group,
  validation_name;